<a href="https://colab.research.google.com/github/koderlad/assignment-M508D/blob/main/M508D_GH1050620.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Assignment started on 30 April 2026.

I am using the dataset from hugging face [https://huggingface.co/datasets/ihmz/enhanced_hiring_data_completeV2]

## **Installing Required Dependencies**
This section is only used to install required dependency.

In [1]:
# !pip install spacy

##**Loading Dependencies**

In [66]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import spacy
import re ##Regular Expression to parse data
from typing import Optional

## **Loading Dataset**

In [60]:
df = pd.read_json('hf://datasets/ihmz/enhanced_hiring_data_completeV2/hiring_chat.json')

In [61]:
df.head()

,conversations
0,"[{'role': 'user', 'content': 'Role: E-Commerce..."
1,"[{'role': 'user', 'content': 'Role: Game Devel..."
2,"[{'role': 'user', 'content': 'Role: Human Reso..."
3,"[{'role': 'user', 'content': 'Role: E-Commerce..."
4,"[{'role': 'user', 'content': 'Role: E-Commerce..."


In [62]:
df_copy = df.copy()

### **REGEX PATTERNS**

In [63]:
FIELD_PATTERN = re.compile(
    r"Role:\s*(?P<role>.*?)\n"
    r"Job Description:\s*(?P<jd>.*?)\n"
    r"Resume_Masked:\s*(?P<resume>.*?)\n"
    r"Transcript_Masked:\s*(?P<transcript>.*)",
    re.DOTALL,
)

In [64]:
SPEAKER_SPLIT = re.compile(r"(Interviewer|\[NAME\]):")

In [65]:
RESUME_SECTION_PATTERN = re.compile(r"\n([A-Z][A-Za-z ]+):\s*\n", re.MULTILINE)

## **PARSING FUNCTIONS**

In [67]:
def parse_user_content(content: str) -> Optional[dict]:
  #First lets split the high-level topics from the conversation.
  m = FIELD_PATTERN.search(content)
  if not m:
    return None
  return {
      'role': m.group('role').strip(),
      "job_description": m.group('jd').strip(),
      "resume": m.group('resume').strip(),
      "transcript": m.group('transcript').strip(),
  }

In [68]:
def parse_transcript_turns(transcript: str) -> list[dict]:
  """Split transcript into speaker turns. Returns list of {speaker, content}."""
  parts = SPEAKER_SPLIT.split(transcript)
  turns = []
  for i in range(1, len(parts) - 1, 2):
    speaker = parts[i]
    content = parts[i + 1].strip()
    if content:
      turns.append({'speaker': speaker, "content": content})
  return turns

In [69]:
def parse_resume_sections(resume: str) -> dict:
  """Split resume by section headers. Returns {section_name_snake: text}."""
  matches = list(RESUME_SECTION_PATTERN.finditer(resume))
  sections = {}
  for i, mh in enumerate(matches):
    name = mh.group(1).strip().lower().replace(" ", "_")
    start = mh.end()
    end = matches[i + 1].start() if i + 1 < len(matches) else len(resume)
    sections[name] = resume[start:end].strip()
  return sections


In [72]:
def normalise_label(raw_label: str) -> str:
    """Lowercase, strip whitespace and punctuation. Customise if labels are noisy."""
    return raw_label.strip().lower().rstrip(".!?\"'")